In [ ]:
from mainnet_launch.constants import AUTO_ETH
from mainnet_launch.pages.solver_diagnostics.solver_rebalance_plans_to_summary_stats import (
    
    DESTINATION_TOKEN_BLOCK_TABLE,
)
from mainnet_launch.database.database_operations import run_read_only_query
import pandas as pd


safe_price_df = run_read_only_query(
    f"""
SELECT DISTINCT safe_price , token_symbol, block_timestamp FROM {DESTINATION_TOKEN_BLOCK_TABLE} where autopool = 'autoETH'

""",
    None,
)

safe_price_df
wide_safe_price_df = safe_price_df.pivot(index="block_timestamp", columns="token_symbol", values="safe_price")
wide_safe_price_df.index = pd.to_datetime(wide_safe_price_df.index, unit='s', utc=True)
wide_safe_price_df = wide_safe_price_df.sort_index()
wide_safe_price_df
# wide_safe_price_df

token_symbol,ETHx,WETH,cbETH,frxETH,osETH,pxETH,rETH,stETH,wstETH
block_timestamp,,,,,,,,,
2024-09-16 20:16:18+00:00,1.038362,1.0,1.07938,0.998768,1.028712,0.996924,1.116761,0.999565,1.178491
2024-09-17 02:16:17+00:00,1.038362,1.0,1.07938,0.998768,1.028712,0.996924,1.116761,0.999565,1.178491
2024-09-17 02:40:37+00:00,1.038362,1.0,1.07938,0.998768,1.028712,0.996924,1.116761,0.999565,1.178491
2024-09-17 02:42:56+00:00,1.038362,1.0,1.07938,0.998768,1.028712,0.996924,1.116761,0.999565,1.178491
2024-09-17 19:40:27+00:00,1.038431,1.0,1.07938,0.994264,1.028896,0.998741,1.117432,0.999565,1.178586
...,...,...,...,...,...,...,...,...,...
2025-04-09 07:30:21+00:00,1.055530,1.0,NaN,NaN,NaN,0.999695,NaN,0.998799,1.197775
2025-04-09 22:30:21+00:00,1.054873,1.0,NaN,NaN,NaN,0.999641,NaN,0.998463,1.197481
2025-04-10 13:30:21+00:00,1.055191,1.0,NaN,NaN,NaN,0.999280,NaN,0.998463,1.197590


In [73]:
asset_allocation_df = run_read_only_query(
    f"""
SELECT amount * portion_ownership * safe_price as autopool_safe_token_value, token_symbol, block_timestamp FROM {DESTINATION_TOKEN_BLOCK_TABLE} where autopool = 'autoETH'
""",
    None,
)
asset_allocation_df =  asset_allocation_df.groupby(['block_timestamp', 'token_symbol']).sum().reset_index()
asset_allocation_df = asset_allocation_df.pivot(index="block_timestamp", columns="token_symbol", values="autopool_safe_token_value")
asset_allocation_df.index = pd.to_datetime(asset_allocation_df.index, unit='s', utc=True)
asset_allocation_df = asset_allocation_df.sort_index()
asset_allocation_df = asset_allocation_df.resample("1d").last()
px.bar(asset_allocation_df)
# wide_safe_price_df

In [58]:
end_of_day_safe_price

token_symbol,ETHx,WETH,cbETH,frxETH,osETH,pxETH,rETH,stETH,wstETH
block_timestamp,,,,,,,,,
2024-09-16 00:00:00+00:00,1.038362,1.0,1.079380,0.998768,1.028712,0.996924,1.116761,0.999565,1.178491
2024-09-17 00:00:00+00:00,1.038431,1.0,1.079380,0.994264,1.028896,0.998741,1.117432,0.999565,1.178586
2024-09-18 00:00:00+00:00,1.038666,1.0,1.079000,0.995110,1.028993,0.998885,1.117600,0.999300,1.178374
2024-09-19 00:00:00+00:00,1.038340,1.0,1.079540,0.994862,1.029120,0.998857,1.117621,0.999285,1.178462
2024-09-20 00:00:00+00:00,1.039542,1.0,1.079388,0.995856,1.029040,0.999135,1.117823,0.999579,1.178913
...,...,...,...,...,...,...,...,...,...
2025-04-07 00:00:00+00:00,1.055475,1.0,NaN,NaN,NaN,0.999448,NaN,0.999147,1.198091
2025-04-08 00:00:00+00:00,1.055305,1.0,NaN,NaN,NaN,0.999475,NaN,0.998799,1.197775
2025-04-09 00:00:00+00:00,1.054873,1.0,NaN,NaN,NaN,0.999641,NaN,0.998463,1.197481


In [ ]:
end_of_day_amount = wide_amount_df.resample('1d').last()
end_of_day_safe_price = wide_safe_price_df.resample('1d').last()[end_of_day_amount.columns]
# not correct does not downscale by the % ownership
safe_value_at_end_of_day = end_of_day_amount.mul(end_of_day_safe_price)
px.line(safe_value_at_end_of_day)

In [46]:

df = run_read_only_query(
    f"""
SELECT safe_price * amount as safe_x_amount, token_symbol, vault_name, block_timestamp FROM {DESTINATION_TOKEN_BLOCK_TABLE} where autopool = 'autoETH'

""",
    None,
)

df.index = pd.to_datetime(df['block_timestamp'], unit='s', utc=True)
df = df.sort_index()
# how do we want to define this

# I think it is the safe value of assets owned at the highest block in the solver

df

,safe_x_amount,token_symbol,vault_name,block_timestamp
block_timestamp,,,,
2024-09-16 20:16:18+00:00,6533.775381,rETH,Balancer rETH Stable Pool,1726517778
2024-09-16 20:16:18+00:00,7099.536085,WETH,Balancer rETH Stable Pool,1726517778
2024-09-16 20:16:18+00:00,1806.354349,pxETH,pxETH/wETH,1726517778
2024-09-16 20:16:18+00:00,1195.354702,WETH,pxETH/wETH,1726517778
2024-09-16 20:16:18+00:00,2080.137467,stETH,pxETH/stETH,1726517778
...,...,...,...,...
2025-04-11 19:30:21+00:00,1279.110147,ETHx,Curve.fi Factory Pool: ETHx-ETH,1744399821
2025-04-11 19:30:21+00:00,495.265570,WETH,Curve.fi Factory Pool: ETHx-ETH,1744399821
2025-04-11 19:30:21+00:00,2715.477670,WETH,Balancer pxETH/wETH StablePool,1744399821


In [ ]:
df.groupby('tok'

In [ ]:

quantity_df = run_read_only_query(
    f"""
SELECT amount FROM {DESTINATION_TOKEN_BLOCK_TABLE} where autopool = 'autoETH'
GROUP BY  token_symbol, block_timestamp

""",
    None,
)
px.line(quantity_df)